## 키워드 검색 후 몇 개의 계정에서 정보를 가지고 올건지

In [ ]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time
import math
import os
import random
import unicodedata
import urllib.request
import urllib
import pandas as pd
import sysm

# Step 2. 사용자에게 필요한 정보들을 입력 받기
print("=" * 80)
print("인스타그램 해쉬태그와 이미지 수집하기")
print("=" * 80)

v_id = input("1.인스타그램의 ID를 입력하세요: ")
if v_id == '':
    v_id = 'seulgi05250'
v_passwd = input("2.인스타그램의 PW를 입력하세요: ")
query_txt = input("3.검색할 해쉬태그를 입력하세요(예: 강남맛집): ")
if query_txt == '':
    query_txt = '강남맛집'

try:
    cnt = int(input('4.수집할 건수는 총 몇 건입니까?(기본값:10): '))
except ValueError:
    cnt = 10
    print('기본값인 10 건으로 수집을 진행합니다.')

page_cnt = math.ceil(cnt / 10)

f_dir = input('5.파일이 저장될 경로만 쓰세요(기본경로 : c:\\py_temp\\ ) : ')
if f_dir == '':
    f_dir = "c:\\py_temp\\"

# Step 3.결과를 저장할 폴더명과 파일명을 설정하고 폴더를 생성하기
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

sec_name = '인스타그램'
img_dir = f_dir + s + '-' + query_txt + '-' + sec_name + '\\' + 'image'

os.makedirs(img_dir)
os.chdir(img_dir)

fc_name = f_dir + s + '-' + query_txt + '-' + sec_name + '\\' + s + '-' + query_txt + '-' + sec_name + '.csv'
fx_name = f_dir + s + '-' + query_txt + '-' + sec_name + '\\' + s + '-' + query_txt + '-' + sec_name + '.xlsx'

# Step 4. 인스타그램 접속 후 자동 로그인 하기
s_time = time.time()
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = "https://www.instagram.com/"
driver.get(url)
driver.maximize_window()

time.sleep(random.randrange(1, 5))

print("\n")
print("요청하신 데이터를 추출중이오니 잠시만 기다려 주세요~~~~^^")
print("\n")

# ID와 비번 입력후 로그인하기
eid = driver.find_element(By.NAME, 'email')
for a in v_id:
    eid.send_keys(a)
    time.sleep(0.3)

epwd = driver.find_element(By.NAME, 'pass')
for b in v_passwd:
    epwd.send_keys(b)
    time.sleep(0.5)

driver.find_element(By.XPATH, '//*[@id="login_form"]/div/div[1]/div/div[3]/div/div/div').click()
time.sleep(5)

# 로그인 정보 나중에 저장하기 클릭
driver.find_element(By.XPATH, '//div[text()="나중에 하기"]').click()
time.sleep(3)
driver.find_element(By.XPATH, '//button[text()="나중에 하기"]').click()
time.sleep(2)

# Step 5. 검색할 키워드 입력하기
element = driver.find_element(By.XPATH, '//*[contains(@aria-label, "검색")]')
element.click()
time.sleep(3)

search_input = driver.find_element(By.XPATH, '//input[@aria-label="입력 검색"]')
for c in query_txt:
    search_input.send_keys(c)
    time.sleep(0.2)

time.sleep(3)

# 왼쪽 리스트 계정 수집 후 몇 개 수집할지 질문
search_results = driver.find_elements(By.XPATH, '//a[@role="link"]')
result_urls = []
for r in search_results:
    href = r.get_attribute('href')
    if href and 'instagram.com' in href and href not in result_urls:
        result_urls.append(href)

try:
    list_cnt = int(input(f'6.현재 {len(result_urls)}개의 계정이 있습니다. 몇 개를 수집하시겠습니까?(기본값:5): '))
except ValueError:
    list_cnt = 5
result_urls = result_urls[:list_cnt]

# Step 6. 전체 게시물의 원본 URL 추출하기
item = []
item2 = []

# 자동 스크롤다운 함수
def scroll_down(driver):
    driver.execute_script("window.scrollTo(0,document.body.scrollHeight);")
    time.sleep(5)

print('요청하신 데이터를 수집중이니 잠시만 기다려 주세요~^^')
print()

count = 1
no2 = []
url2 = []
hash2 = []

for idx, result_url in enumerate(result_urls):
    print('= %s번째 계정 수집 시작 ========================================' % (idx + 1))

    driver.get(result_url)
    time.sleep(random.randrange(3, 6))

    item = []
    item2 = []
    a = 1

    while (a <= page_cnt):
        scroll_down(driver)

        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')

        all_a = soup.find('div', 'x1n2onr6').find_all('a')
        for i in all_a:
            href = i.get('href')
            if href and ('/p/' in href or '/reel/' in href):
                item.append(href)
        item2 = pd.Series(item).drop_duplicates()

        if len(item2) >= cnt:
            break

        a += 1
        print('요청하신 데이터를 수집중이니 잠시만 더 기다려 주세요~^^')

    # 추출된 URL 사용하여 전체 URL 완성하기
    full_url = []
    url_cnt = 1

    print('= 수집될 인스타그램 주소는 아래와 같습니다 =========')

    for x in item2:
        url = 'https://www.instagram.com' + x
        full_url.append(url)
        print(url_cnt, ':', url)
        if url_cnt >= cnt:
            break
        url_cnt += 1

    print('========================================')
    print()

    # Step 7. 각 페이지별로 이미지와 해쉬태그를 수집하기
    for c in full_url:
        print()
        driver.get(c)
        time.sleep(random.randrange(3, 9))

        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')

        tags_1 = soup.find_all('a', href=lambda x: x and '/explore/tags/' in x)

        if len(tags_1) == 0:
            print('해시태그 없음, 건너뜀')
            continue

        print('%s번째 게시물의 대표 이미지와 해쉬태그를 수집합니다~~~' % count)
        print('게시물 URL:', c)

        no2.append(count)
        url2.append(c)

        # img_src = soup.find('div', '_aagv').find('img')['src']
        img_tag = soup.find('div', '_aagv') or soup.find('div', '_aagw')
        img_src = img_tag.find('img')['src']
        urllib.request.urlretrieve(img_src, str(count) + '.jpg')
        print(img_dir, '아래에 %s번째 이미지 저장 완료===' % count)

        bmp_map = dict.fromkeys(range(0x10000, sys.maxunicode + 1), 0xfffd)
        hash_tags = []

        for d in tags_1:
            tags = d.get_text()
            tags_11 = tags.translate(bmp_map)
            tags_2 = unicodedata.normalize('NFC', tags_11)
            if tags_2[0:1] == '#':
                hash_tags.append(tags_2)

        print(hash_tags)
        hash2.append(hash_tags)
        count += 1

# Step 8. 수집된 해시태그를 csv , xls 형식으로 저장하기
insta = pd.DataFrame()
insta['번호'] = no2
insta['URL주소'] = url2
insta['해쉬태그'] = pd.Series(hash2)

insta.to_csv(fc_name, encoding="utf-8-sig", index=False)
insta.to_excel(fx_name, index=False, engine='openpyxl')

# Step 9. 요약 정보 출력하기
e_time = time.time()
t_time = e_time - s_time

print("=" * 120)
print("1.총 소요시간: %s 초" % round(t_time, 1))
print("2.총 저장 건수: %s 건 " %(count - 1))
print("3.csv파일 저장 경로: %s" % fc_name)
print("4.xls파일 저장 경로: %s" % fx_name)
print("5.이미지파일 저장 경로: %s" % img_dir)
print("=" * 120)

인스타그램 해쉬태그와 이미지 수집하기


요청하신 데이터를 추출중이오니 잠시만 기다려 주세요~~~~^^


요청하신 데이터를 수집중이니 잠시만 기다려 주세요~^^

= 1번째 계정 수집 시작 ========================================
= 수집될 인스타그램 주소는 아래와 같습니다 =========
1 : https://www.instagram.com/tasty.gangnam/p/DUhQXvikV-t/
2 : https://www.instagram.com/tasty.gangnam/p/DUZ9cLoEXrl/
3 : https://www.instagram.com/tasty.gangnam/reel/DVH4UtvEuuE/
4 : https://www.instagram.com/tasty.gangnam/p/DVpWm_WEV_4/
5 : https://www.instagram.com/tasty.gangnam/reel/DViTR6dEYOz/


1번째 게시물의 대표 이미지와 해쉬태그를 수집합니다~~~
게시물 URL: https://www.instagram.com/tasty.gangnam/p/DUhQXvikV-t/
c:\py_temp\2026-03-09-15-57-01-강남맛집-인스타그램\image 아래에 1번째 이미지 저장 완료===
['#까폼', '#강남태국음식', '#태국로컬맛집', '#팟타이맛집', '#강남맛집']

2번째 게시물의 대표 이미지와 해쉬태그를 수집합니다~~~
게시물 URL: https://www.instagram.com/tasty.gangnam/p/DUZ9cLoEXrl/
c:\py_temp\2026-03-09-15-57-01-강남맛집-인스타그램\image 아래에 2번째 이미지 저장 완료===
['#서울벚꽃', '#벚꽃시즌', '#2026벚꽃', '#벚꽃개화시기']

3번째 게시물의 대표 이미지와 해쉬태그를 수집합니다~~~
게시물 URL: https://www.instagram.com/tasty.gangnam/reel/DVH4Ut

## 키워드 검색 후 몇 개의 계정에서 정보를 가지고 올건지

## 최종 수정본 

In [2]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time
import math
import os
import random
import unicodedata
import urllib.request
import urllib
import pandas as pd
import sys

# Step 2. 사용자에게 필요한 정보들을 입력 받기
print("=" * 80)
print("인스타그램 해쉬태그와 이미지 수집하기")
print("=" * 80)

v_id = input("1.인스타그램의 ID를 입력하세요: ")
if v_id == '':
    v_id = 'yeonjudata'
v_passwd = input("3.검색할 해쉬태그를 입력하세요(예: 강남맛집): ")
query_txt = input("3.검색할 해쉬태그를 입력하세요(예: 강남맛집): ")
if query_txt == '':
    query_txt = '강남맛집'

try:
    cnt = int(input('4.수집할 건수는 총 몇 건입니까?(기본값:10): '))
except ValueError:
    cnt = 10
    print('기본값인 10 건으로 수집을 진행합니다.')

page_cnt = math.ceil(cnt / 10)

f_dir = input('5.파일이 저장될 경로만 쓰세요(기본경로 : c:\\py_temp\\ ) : ')
if f_dir == '':
    f_dir = "c:\\py_temp\\"

# Step 3.결과를 저장할 폴더명과 파일명을 설정하고 폴더를 생성하기
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

sec_name = '인스타그램'
img_dir = f_dir + s + '-' + query_txt + '-' + sec_name + '\\' + 'image'

os.makedirs(img_dir)
os.chdir(img_dir)

fc_name = f_dir + s + '-' + query_txt + '-' + sec_name + '\\' + s + '-' + query_txt + '-' + sec_name + '.csv'
fx_name = f_dir + s + '-' + query_txt + '-' + sec_name + '\\' + s + '-' + query_txt + '-' + sec_name + '.xlsx'

# Step 4. 인스타그램 접속 후 자동 로그인 하기
s_time = time.time()
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = "https://www.instagram.com/"
driver.get(url)
driver.maximize_window()

time.sleep(random.randrange(1, 5))

print("\n")
print("요청하신 데이터를 추출중이오니 잠시만 기다려 주세요~~~~^^")
print("\n")

# ID와 비번 입력후 로그인하기
eid = driver.find_element(By.NAME, 'email')
for a in v_id:
    eid.send_keys(a)
    time.sleep(0.3)

epwd = driver.find_element(By.NAME, 'pass')
for b in v_passwd:
    epwd.send_keys(b)
    time.sleep(0.5)

driver.find_element(By.XPATH, '//*[@id="login_form"]/div/div[1]/div/div[3]/div/div/div').click()
time.sleep(10)

# 로그인 정보 나중에 저장하기 클릭
driver.find_element(By.XPATH, '//div[text()="나중에 하기"]').click()
time.sleep(3)
driver.find_element(By.XPATH, '//button[text()="나중에 하기"]').click()
time.sleep(2)

# Step 5. 검색할 키워드 입력하기
element = driver.find_element(By.XPATH, '//*[contains(@aria-label, "검색")]')
element.click()
time.sleep(3)

search_input = driver.find_element(By.XPATH, '//input[@aria-label="입력 검색"]')
for c in query_txt:
    search_input.send_keys(c)
    time.sleep(0.2)

time.sleep(3)

# 왼쪽 리스트 계정 수집 후 몇 개 수집할지 질문
search_results = driver.find_elements(By.XPATH, '//a[@role="link"]')
result_urls = []
for r in search_results:
    href = r.get_attribute('href')
    if href and 'instagram.com' in href and href not in result_urls:
        result_urls.append(href)

try:
    list_cnt = int(input(f'6.현재 {len(result_urls)}개의 계정이 있습니다. 몇 개를 수집하시겠습니까?(기본값:5): '))
except ValueError:
    list_cnt = 5
result_urls = result_urls[:list_cnt]

# Step 6. 전체 게시물의 원본 URL 추출하기
item = []
item2 = []

# 자동 스크롤다운 함수
def scroll_down(driver):
    driver.execute_script("window.scrollTo(0,document.body.scrollHeight);")
    time.sleep(5)

print('요청하신 데이터를 수집중이니 잠시만 기다려 주세요~^^')
print()

count = 1
no2 = []
url2 = []
hash2 = []

for idx, result_url in enumerate(result_urls):
    print('= %s번째 계정 수집 시작 ========================================' % (idx + 1))

    driver.get(result_url)
    time.sleep(random.randrange(3, 6))

    item = []
    item2 = []
    a = 1

    while (a <= page_cnt):
        scroll_down(driver)

        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')

        all_a = soup.find('div', 'x1n2onr6').find_all('a')
        for i in all_a:
            href = i.get('href')
            if href and ('/p/' in href or '/reel/' in href):
                item.append(href)
        item2 = pd.Series(item).drop_duplicates()

        if len(item2) >= cnt:
            break

        a += 1
        print('요청하신 데이터를 수집중이니 잠시만 더 기다려 주세요~^^')

    # 추출된 URL 사용하여 전체 URL 완성하기
    full_url = []
    url_cnt = 1

    print('= 수집될 인스타그램 주소는 아래와 같습니다 =========')

    for x in item2:
        url = 'https://www.instagram.com' + x
        full_url.append(url)
        print(url_cnt, ':', url)
        if url_cnt >= cnt:
            break
        url_cnt += 1

    print('========================================')
    print()

    # Step 7. 각 페이지별로 이미지와 해쉬태그를 수집하기
    for c in full_url:
        print()
        driver.get(c)
        time.sleep(random.randrange(3, 9))

        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')

        tags_1 = soup.find_all('a', href=lambda x: x and '/explore/tags/' in x)

        if len(tags_1) == 0:
            print('해시태그 없음, 건너뜀')
            continue

        print('%s번째 게시물의 대표 이미지와 해쉬태그를 수집합니다~~~' % count)
        print('게시물 URL:', c)

        no2.append(count)
        url2.append(c)

        try:
            if '/reel/' in c:
                img_src = soup.find('meta', property='og:image')['content']
            else:img_src = soup.find('div', '_aagv').find('img')['src']
        except:
            print('%s번째 이미지 저장 실패, 건너뜀' % count)
            no2.pop()
            url2.pop()
            continue
        urllib.request.urlretrieve(img_src, str(count) + '.jpg')
        print(img_dir, '아래에 %s번째 이미지 저장 완료===' % count)

        bmp_map = dict.fromkeys(range(0x10000, sys.maxunicode + 1), 0xfffd)
        hash_tags = []

        for d in tags_1:
            tags = d.get_text()
            tags_11 = tags.translate(bmp_map)
            tags_2 = unicodedata.normalize('NFC', tags_11)
            if tags_2[0:1] == '#':
                hash_tags.append(tags_2)

        print(hash_tags)
        hash2.append(hash_tags)
        count += 1

# Step 8. 수집된 해시태그를 csv , xls 형식으로 저장하기
insta = pd.DataFrame()
insta['번호'] = no2
insta['URL주소'] = url2
insta['해쉬태그'] = pd.Series(hash2)

insta.to_csv(fc_name, encoding="utf-8-sig", index=False)
insta.to_excel(fx_name, index=False, engine='openpyxl')

# Step 9. 요약 정보 출력하기
e_time = time.time()
t_time = e_time - s_time

print("=" * 120)
print("1.총 소요시간: %s 초" % round(t_time, 1))
print("2.총 저장 건수: %s 건 " %(count - 1))
print("3.csv파일 저장 경로: %s" % fc_name)
print("4.xls파일 저장 경로: %s" % fx_name)
print("5.이미지파일 저장 경로: %s" % img_dir)
print("=" * 120)

인스타그램 해쉬태그와 이미지 수집하기




요청하신 데이터를 추출중이오니 잠시만 기다려 주세요~~~~^^


요청하신 데이터를 수집중이니 잠시만 기다려 주세요~^^

= 1번째 계정 수집 시작 ========================================
= 수집될 인스타그램 주소는 아래와 같습니다 =========
1 : https://www.instagram.com/tasty.gangnam/p/DUhQXvikV-t/
2 : https://www.instagram.com/tasty.gangnam/p/DUZ9cLoEXrl/
3 : https://www.instagram.com/tasty.gangnam/reel/DVH4UtvEuuE/
4 : https://www.instagram.com/tasty.gangnam/reel/DVp_1skCSUt/
5 : https://www.instagram.com/tasty.gangnam/p/DVpWm_WEV_4/


1번째 게시물의 대표 이미지와 해쉬태그를 수집합니다~~~
게시물 URL: https://www.instagram.com/tasty.gangnam/p/DUhQXvikV-t/
c:\py_temp\2026-03-09-17-59-33-강남맛집-인스타그램\image 아래에 1번째 이미지 저장 완료===
['#까폼', '#강남태국음식', '#태국로컬맛집', '#팟타이맛집', '#강남맛집']

2번째 게시물의 대표 이미지와 해쉬태그를 수집합니다~~~
게시물 URL: https://www.instagram.com/tasty.gangnam/p/DUZ9cLoEXrl/
c:\py_temp\2026-03-09-17-59-33-강남맛집-인스타그램\image 아래에 2번째 이미지 저장 완료===
['#서울벚꽃', '#벚꽃시즌', '#2026벚꽃', '#벚꽃개화시기']

3번째 게시물의 대표 이미지와 해쉬태그를 수집합니다~~~
게시물 URL: https://www.instagram.com/tasty.gangnam/reel/DVH4UtvEuuE/
c:\py_temp\202